# 03 · Chatbot demo - medical, medication, app, patient and safety questions

**Objective.** Use the assistant the way the app will: in English, Modern Standard Arabic and Saudi dialect, with and
without a synthetic patient profile, through Python, the FastAPI endpoint and the Gradio UI.

> Patient profiles are **synthetic test data**. Never enter real patient information.

**Patient mode vs developer mode.** `show()` renders the patient view exactly as the Gradio chat shows it (no status, no `[E#]` markers, short source line for medical answers). `show(q, developer=True)` adds the technical view; the API's `answer` keeps the citation markers.

In [ ]:
# Stage 0 - setup: find the project root, make `src/` importable, load .env (keys are never printed)
import sys, os, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "vitala_rag").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Open this notebook from inside the VITALA_RAG project (could not find src/vitala_rag).")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env", override=False)
except ImportError:
    pass

import pandas as pd
pd.set_option("display.max_colwidth", 120)
from vitala_rag.config import Paths
paths = Paths(root=ROOT)
print("Project root:", ROOT)
print("LLM provider configured:", os.environ.get("VITALA_LLM_PROVIDER", "(not set)"), "| API keys are not displayed")

In [ ]:
from vitala_rag.chatbot import VitalaRAG, get_llm
from vitala_rag.retrieval import Embedder
PROVIDER = os.environ.get("VITALA_LLM_PROVIDER", "extractive") if (os.environ.get("AZURE_OPENAI_API_KEY")
                                                                    or os.environ.get("ANTHROPIC_API_KEY")) else "extractive"
rag = VitalaRAG(paths=paths, llm=get_llm(PROVIDER), embedder=Embedder(), translate_queries=PROVIDER != "extractive", log=False)
print("provider:", rag.llm.name, "| profiles:", [p["patient_id"] for p in rag.list_patient_profiles()])

def show(q, patient=None, debug=False):
    r = rag.answer(q, patient_id=patient, debug=debug)
    print(f"Q: {q}" + (f"   [profile {patient}]" if patient else ""))
    print(f"-> {r['status']} | route {r['route']} | {r['timings_ms'].get('total')} ms | LLM calls {r['llm_usage']['calls']}\n")
    print(r["answer"])
    for c in r["citations"]:
        print(f"   [{c['evidence_id']}] {c['citation'][:140]}")
    print("-" * 100)
    return r

## 1 · Preventive and medication questions (English, MSA, Saudi dialect)

In [ ]:
show("When should adults start colorectal cancer screening?")
show("هل يسبب الليسينوبريل كحة؟")
_ = show("وش أسوي بالميتفورمين في رمضان؟")

## 2 · VITALA app questions (answered only from the app documentation, with an availability note)

In [ ]:
show("كيف ألغي موعدي؟")
_ = show("Can the clinic see my health information?")

## 3 · Synthetic patient context

Patient facts are rendered **by code** from the profile and cited as `[P#]`; the LLM only adds the general medical part.
Missing data is reported as missing (profile C has no HbA1c).

In [ ]:
show("What was my most recent HbA1c?", "SYN-DEMO-A")
show("What was my latest HbA1c result?", "SYN-DEMO-C")
_ = show("Why is HbA1c in my screening plan?", "SYN-DEMO-A")

## 4 · Safety behaviour (rules run before any retrieval or LLM call)

In [ ]:
for q in ["My lips are swelling after my new lisinopril", "Should I stop taking lisinopril because of my cough?",
          "What are the side effects of amlodipine?", "How do I treat a migraine attack?"]:
    show(q)

## 5 · Developer debug view (never for patients)

The trace shows the detected language, route, retrieved chunks with scores, the evidence placed in the prompt, raw LLM
output, grounding checks and stage latencies.

In [ ]:
r = rag.answer("Is it a problem to take ibuprofen together with lisinopril?", debug=True)
d = r["debug"]
print("language:", d["detected_language"], "| route:", d["final"]["route"], "| timings:", d["timings_ms"])
display(pd.DataFrame(d["retrieved"])[["rank", "chunk_id", "dense_score", "bm25_score", "rrf"]])
print("evidence in prompt:", [e["chunk_id"] for e in d.get("evidence_in_prompt", [])])
print("grounding:", [(g["attempt"], g["verdict"], g["reasons"]) for g in d.get("grounding_checks", [])])

## 6 · FastAPI endpoint (in-process test client)

Run the real server from a terminal with `python -m vitala_rag api` (http://127.0.0.1:8000/docs). Here the same app is
called in-process.

In [ ]:
from fastapi.testclient import TestClient
from vitala_rag.interfaces import app, set_rag
set_rag(rag)
client = TestClient(app)
print(client.get("/health").json())
resp = client.post("/v1/chat", json={"question": "Why do ACE inhibitors cause a dry cough?", "debug": True}).json()
print(resp["status"], "| debug returned to client:", resp.get("debug") is not None, "(only with VITALA_DEBUG=1)")
print(resp["answer"][:400])

## 7 · Gradio UI

Set `LAUNCH = True` to open the chat UI on http://127.0.0.1:7860 (local only, no public link). `developer=True` adds the
diagnostics panel. Stop it with `demo.close()`.

In [ ]:
LAUNCH = False
if LAUNCH:
    from vitala_rag.interfaces import build_demo
    demo = build_demo(rag, developer=True)
    demo.launch(server_name="127.0.0.1", server_port=7860, share=False, prevent_thread_lock=True)